# YuvaIntern Task 5 — Analytics Report & Insights Documentation
## Business Performance Analysis using the Tips Dataset

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** Seaborn's public `tips` teaching dataset  
**Purpose:** Demonstrate a structured reporting workflow: business objectives, data analysis, visualizations, findings, executive summary, and actionable recommendations.

**Important context:** This is a teaching dataset from a restaurant setting, not a company's confidential operational data. Recommendations are exploratory and should be validated against a restaurant's own data before being used for real business decisions.

## 1. Business objective and analytical questions

### Business objective
Explore the available restaurant bill and tipping records to understand bill amounts, tips, party sizes, day/time patterns, and differences across customer and service groups.

### Questions
1. What are the overall bill and tip patterns?
2. How are tip amounts related to total bills?
3. Do average tips or bills vary by day and meal time?
4. How do party size and smoking-table indicator relate to bill/tip measures?
5. Which findings can be converted into cautious, testable operational recommendations?

### Metrics
- Average and median total bill
- Average and median tip
- Tip as a percentage of total bill
- Number of records and party size
- Group-level averages by day, time, and party size

In [ ]:
# Install if needed:
# %pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Load a commonly used public teaching dataset from Seaborn.
# If internet is unavailable, Seaborn may need the dataset cached or accessible online.
df = sns.load_dataset("tips")

print("Shape:", df.shape)
display(df.head())


## 2. Data understanding and quality checks

In [ ]:
print("Columns:", df.columns.tolist())
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(quality)
print("Exact duplicate rows:", df.duplicated().sum())


## 3. Data preparation and metric creation

The dataset is checked before analysis. Missing values, if present, are reported rather than automatically imputed. Exact duplicate rows are reviewed, but the notebook does not blindly delete them because two legitimate restaurant bills can share the same field values in a teaching dataset. A tip percentage metric is created for exploratory comparison.

In [ ]:
analysis = df.copy()

# Keep a documented count of exact duplicate rows before any optional removal.
duplicate_count = int(analysis.duplicated().sum())
print("Exact duplicate rows found:", duplicate_count)

# Create tip percentage, guarding against zero total bills.
analysis["tip_percent"] = np.where(
    analysis["total_bill"] > 0,
    (analysis["tip"] / analysis["total_bill"]) * 100,
    np.nan
)

# No rows are removed automatically: missingness and duplicates are first assessed.
print("Rows retained for descriptive analysis:", len(analysis))
display(analysis.head())


## 4. Executive metrics

In [ ]:
executive_metrics = pd.Series({
    "Number of records": len(analysis),
    "Average total bill": analysis["total_bill"].mean(),
    "Median total bill": analysis["total_bill"].median(),
    "Average tip": analysis["tip"].mean(),
    "Median tip": analysis["tip"].median(),
    "Average tip percentage": analysis["tip_percent"].mean(),
    "Average party size": analysis["size"].mean(),
    "Missing cells": int(analysis.isna().sum().sum()),
    "Exact duplicate rows": duplicate_count
}, name="Value")
display(executive_metrics.to_frame())


## 5. Analysis by day and meal time

In [ ]:
day_summary = (
    analysis.groupby("day", observed=False)
    .agg(
        records=("total_bill", "size"),
        avg_bill=("total_bill", "mean"),
        median_bill=("total_bill", "median"),
        avg_tip=("tip", "mean"),
        avg_tip_percent=("tip_percent", "mean"),
        avg_party_size=("size", "mean")
    ).sort_values("avg_bill", ascending=False)
)
display(day_summary.round(2))

time_summary = (
    analysis.groupby("time", observed=False)
    .agg(
        records=("total_bill", "size"),
        avg_bill=("total_bill", "mean"),
        avg_tip=("tip", "mean"),
        avg_tip_percent=("tip_percent", "mean")
    ).sort_values("avg_bill", ascending=False)
)
display(time_summary.round(2))


## 6. Visualizations

In [ ]:
# Chart 1: Average bill by day
day_bill = analysis.groupby("day", observed=False)["total_bill"].mean()
plt.figure(figsize=(8, 5))
plt.bar(day_bill.index.astype(str), day_bill.values)
plt.title("Average Total Bill by Day")
plt.xlabel("Day")
plt.ylabel("Average total bill")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 2: Total bill distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis["total_bill"].dropna(), bins=20, edgecolor="black")
plt.title("Distribution of Total Bills")
plt.xlabel("Total bill")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 3: Total bill vs tip
plt.figure(figsize=(8, 5))
sns.scatterplot(data=analysis, x="total_bill", y="tip", hue="time")
plt.title("Total Bill vs Tip by Meal Time")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 4: Tip percentage by party size
party_summary = analysis.groupby("size", observed=False)["tip_percent"].mean()
plt.figure(figsize=(8, 5))
plt.plot(party_summary.index, party_summary.values, marker="o")
plt.title("Average Tip Percentage by Party Size")
plt.xlabel("Party size")
plt.ylabel("Average tip (%)")
plt.tight_layout()
plt.show()


## 7. Relationships and group comparisons

In [ ]:
numeric_corr = analysis[["total_bill", "tip", "size", "tip_percent"]].corr()
display(numeric_corr.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(numeric_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Bill, Tip, Party Size and Tip Percentage")
plt.tight_layout()
plt.show()


In [ ]:
# Compare bills and tips by smoker indicator, without implying causation.
smoker_summary = (
    analysis.groupby("smoker", observed=False)
    .agg(
        records=("total_bill", "size"),
        avg_bill=("total_bill", "mean"),
        avg_tip=("tip", "mean"),
        avg_tip_percent=("tip_percent", "mean")
    )
)
display(smoker_summary.round(2))


## 8. Data-driven findings and recommendations

The next cell creates findings from the calculated data. Interpret differences as associations, not proof that one factor causes another. A small teaching dataset may not represent a whole restaurant, region, or current market.

In [ ]:
# Findings generated from current dataset calculations
highest_bill_day = day_summary["avg_bill"].idxmax()
lowest_bill_day = day_summary["avg_bill"].idxmin()
highest_bill_time = time_summary["avg_bill"].idxmax()
corr_bill_tip = analysis[["total_bill", "tip"]].corr().iloc[0, 1]
largest_party = int(analysis["size"].max())

print(f"Day with highest average bill in this dataset: {highest_bill_day}")
print(f"Day with lowest average bill in this dataset: {lowest_bill_day}")
print(f"Meal time with highest average bill in this dataset: {highest_bill_time}")
print(f"Correlation between total bill and tip: {corr_bill_tip:.3f}")
print(f"Largest recorded party size: {largest_party}")

print("\nSuggested actions to validate with real operational data:")
print("1. Compare day/time patterns with staffing and booking records before changing schedules.")
print("2. Track average bill, party size, and tip percentage together rather than using one metric alone.")
print("3. Investigate unusually high bills and tips as individual records before drawing conclusions.")
print("4. Repeat the analysis on a longer, representative dataset before making policy decisions.")


## 9. Executive summary

This notebook evaluates restaurant bill and tipping records using descriptive statistics, grouped summaries, and visualizations. The calculated tables and charts summarize bill size, tips, party size, day/time patterns, and associations between numeric variables. The exact numerical results are generated when the notebook runs and should be copied from the outputs into a final presentation only after the notebook has been executed.

## 10. Limitations

- The dataset is a small teaching dataset and is not guaranteed to represent a real restaurant's full operations.
- The data are observational; associations do not establish causation.
- The dataset does not provide costs, margins, staffing hours, customer satisfaction, or booking capacity, so profit and staffing conclusions cannot be directly calculated.
- Recommendations should be validated with a larger, current, business-specific dataset.

## 11. Conclusion

The project demonstrates a reporting workflow that links business questions to metrics, analysis, charts, interpretation, an executive summary, and practical recommendations. It emphasizes transparent data-quality checks and cautious interpretation.